# Medical Conversational AI — EDA & Preprocessing Checkpoint

**Pipeline**: NLU -> NER -> Fine-tuning -> NLG

**Datasets**: MedQuAD (16k QA pairs) + PubMed abstracts

**Sections**: Setup | MedQuAD Load | MedQuAD EDA | PubMed Fetch | PubMed EDA | Preprocessing | Baseline NER | Summary

---
## 1. Setup

In [ ]:
# ── optional installs (run once) ────────────────────────────────────────────
# !pip install biopython scispacy
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz

import os, re, json, time, warnings, random
from pathlib import Path
from collections import Counter

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for _pkg in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(_pkg, quiet=True)
from nltk.corpus   import stopwords as _sw
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
random.seed(42);  np.random.seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (14, 5), 'font.size': 11})

STOP_WORDS = set(_sw.words('english'))

# ── paths ────────────────────────────────────────────────────────────────────
RAW_DIR       = Path('../data/raw')
PROC_DIR      = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

MEDQUAD_CSV = RAW_DIR / 'medquad.csv'
PUBMED_CSV  = RAW_DIR / 'pubmed_abstracts.csv'

print("✅ Imports ready")
print(f"   Raw dir      : {RAW_DIR.resolve()}")
print(f"   Processed dir: {PROC_DIR.resolve()}")


---
## 2. MedQuAD — Load & Parse

Loads the CSV (question, nswer, source, ocus_area) and derives question_type via keyword heuristic. XML loader is commented in for the raw MedQuAD GitHub repo.

In [ ]:
# ── 2a. Load CSV ─────────────────────────────────────────────────────────────
# The CSV has columns: question, answer, source, focus_area
# (No question_type column — we derive it from the question wording.)
mq = pd.read_csv(MEDQUAD_CSV)
print(f"Raw shape : {mq.shape}")
print(f"Columns   : {mq.columns.tolist()}")
mq.head(3)


In [ ]:
# ── 2b. Infer question_type from question text ───────────────────────────────
# MedQuAD questions follow predictable templates; this regex-based heuristic
# captures the main intent categories that matter for our NLU stage.

def infer_question_type(q: str) -> str:
    """Map a question string to one of 10 intent categories."""
    q = str(q).lower().strip()
    if re.search(r"\bwhat is\b|\bwhat are\b|\bdefine\b", q):
        return "definition"
    if re.search(r"\bcaus|\bwhy\b|\brisk factor", q):
        return "causes"
    if re.search(r"\bsymptom|\bsign\b|\bmanifest", q):
        return "symptoms"
    if re.search(r"\btreat|\btherapy|\bmanage|\bmedication|\bdrug\b", q):
        return "treatment"
    if re.search(r"\bdiagnos", q):
        return "diagnosis"
    if re.search(r"\bprevent", q):
        return "prevention"
    if re.search(r"\binherit|\bgenetic|\bchromosom", q):
        return "inheritance"
    if re.search(r"\bhow many|\bhow common|\bprevalence|\bfrequency", q):
        return "frequency"
    if re.search(r"\bwho.*risk|\bat risk|\bsuscepti", q):
        return "susceptibility"
    if re.search(r"\bresearch|\bclinical trial|\bstudy", q):
        return "research"
    return "other"

mq["question_type"] = mq["question"].apply(infer_question_type)

# Rename for clarity & consistency throughout the notebook
mq = mq.rename(columns={"focus_area": "focus"})

print("question_type distribution:")
print(mq["question_type"].value_counts().to_string())


In [ ]:
# ── 2c. If you have the raw MedQuAD XML instead of the CSV ──────────────────
# Uncomment the block below to load directly from the GitHub XML structure.
#
# from xml.etree import ElementTree as ET
# records = []
# for xml_path in sorted(Path("../data/raw/MedQuAD").rglob("*.xml")):
#     tree = ET.parse(xml_path)
#     root = tree.getroot()
#     source = root.attrib.get("Source", "")
#     for qa in root.findall(".//QAPair"):
#         question = (qa.findtext("Question") or "").strip()
#         answer   = (qa.findtext("Answer")   or "").strip()
#         qtype    = qa.find("Question").attrib.get("qtype", "other") if qa.find("Question") is not None else "other"
#         focus    = root.findtext(".//Focus") or ""
#         records.append({"question": question, "answer": answer,
#                         "question_type": qtype, "focus": focus, "source": source})
# mq = pd.DataFrame(records)
# print("Loaded from XML:", mq.shape)
#
# ─── What to check after loading ────────────────────────────────────────────
# 1. Run mq.isnull().sum() — expect very few nulls; drop rows where question/answer is NaN.
# 2. Run mq["question_type"].value_counts() — official XML has richer qtype labels.
# 3. Run mq["source"].value_counts() — 12 NIH sub-sources expected.
print("XML loader ready (commented out — using CSV path)")


---
## 3. MedQuAD — EDA

- question_type distribution (flag classes < 50)
- Top-20 focus areas
- Q/A length histograms
- Missing/malformed entries
- 10 random QA pairs

In [ ]:
# ── 3a. question_type distribution (bar chart) ───────────────────────────────
qt_counts = mq["question_type"].value_counts()
LOW_COUNT_THRESHOLD = 50

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#d62728" if v < LOW_COUNT_THRESHOLD else "#1f77b4" for v in qt_counts.values]
bars = ax.bar(qt_counts.index, qt_counts.values, color=colors, edgecolor="white")
ax.set_title("MedQuAD — Question Type Distribution\n(red = fewer than 50 examples)", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=35)
for bar, val in zip(bars, qt_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            str(val), ha="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(PROC_DIR / "mq_qtype_dist.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Flag sparse classes ──────────────────────────────────────────────────────
sparse = qt_counts[qt_counts < LOW_COUNT_THRESHOLD]
if sparse.empty:
    print(f"✅  No question_type has fewer than {LOW_COUNT_THRESHOLD} examples.")
else:
    print(f"⚠️  Classes with < {LOW_COUNT_THRESHOLD} examples (need oversampling / class weights):")
    print(sparse.to_string())


In [ ]:
# ── 3b. Top-20 focus / disease areas ─────────────────────────────────────────
# The "focus" column contains the disease or drug name each QA pair addresses.
focus_counts = mq["focus"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(x=focus_counts.values, y=focus_counts.index, palette="viridis", ax=ax)
ax.set_title("Top 20 Focus Areas (disease / drug) in MedQuAD", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of QA pairs")
for i, val in enumerate(focus_counts.values):
    ax.text(val + 0.3, i, str(val), va="center", fontsize=9)
plt.tight_layout()
plt.savefig(PROC_DIR / "mq_top20_focus.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Total unique focus areas: {mq['focus'].nunique()}")


In [ ]:
# ── 3c. Question & answer length distributions ───────────────────────────────
# Word counts are the most interpretable metric for a QA system.
mq["q_words"] = mq["question"].apply(lambda x: len(str(x).split()))
mq["a_words"] = mq["answer"].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(mq["q_words"], bins=40, color="#2196F3", edgecolor="white", alpha=0.85)
axes[0].set_title("Question Length (word count)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Words"); axes[0].set_ylabel("Frequency")
axes[0].axvline(mq["q_words"].median(), color="red", linestyle="--", label=f"median={mq['q_words'].median():.0f}")
axes[0].legend()

axes[1].hist(mq["a_words"], bins=60, color="#4CAF50", edgecolor="white", alpha=0.85)
axes[1].set_title("Answer Length (word count)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Words"); axes[1].set_ylabel("Frequency")
axes[1].axvline(mq["a_words"].median(), color="red", linestyle="--", label=f"median={mq['a_words'].median():.0f}")
# Clip x-axis at 99th percentile so one giant answer doesn't squash the plot
p99 = int(mq["a_words"].quantile(0.99))
axes[1].set_xlim(0, p99)
axes[1].legend()

plt.suptitle("MedQuAD — Length Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PROC_DIR / "mq_length_dist.png", dpi=150, bbox_inches="tight")
plt.show()

print("Question length stats (words):")
print(mq["q_words"].describe().round(1))
print("\nAnswer length stats (words):")
print(mq["a_words"].describe().round(1))


In [ ]:
# ── 3d. Missing values & malformed entries ───────────────────────────────────
print("=== Missing values ===")
print(mq[["question", "answer", "source", "focus", "question_type"]].isnull().sum())

# A "malformed" answer is one that is null, empty, or suspiciously short (<5 words)
missing_ans  = mq["answer"].isnull().sum()
empty_ans    = (mq["answer"].fillna("").str.strip() == "").sum()
short_ans    = (mq["a_words"] < 5).sum()
missing_focus= mq["focus"].isnull().sum()

print(f"\n=== Malformed entries ===")
print(f"  Missing answers      : {missing_ans}")
print(f"  Empty answers        : {empty_ans}")
print(f"  Very short answers   : {short_ans}  (< 5 words)")
print(f"  Missing focus field  : {missing_focus}")

# Show a few suspect rows
suspect = mq[mq["a_words"] < 5].head(5)
if not suspect.empty:
    print("\nSample short/malformed answers:")
    for _, row in suspect.iterrows():
        print(f"  [{row['source']}] Q: {str(row['question'])[:60]} → A: {str(row['answer'])[:60]}")


In [ ]:
# ── 3e. 10 random QA pairs for manual inspection ────────────────────────────
# Reading real examples helps catch data quality issues that stats miss.
print("=" * 80)
print("10 RANDOM QA PAIRS — manual inspection")
print("=" * 80)
sample_rows = mq.dropna(subset=["answer"]).sample(10, random_state=7)
for i, (_, row) in enumerate(sample_rows.iterrows(), 1):
    print(f"\n[{i}] Source={row['source']} | Type={row['question_type']} | Focus={row['focus']}")
    print(f"  Q: {row['question']}")
    # Truncate very long answers for readability
    ans_preview = str(row["answer"])[:300] + ("..." if len(str(row["answer"])) > 300 else "")
    print(f"  A: {ans_preview}")
print("\n" + "=" * 80)


---
## 4. PubMed — Fetch via NCBI Entrez

Pulls 2,000-5,000 abstracts keyed to the top MedQuAD focus terms. Results cached to data/processed/pubmed_fetched.csv. Replace YOUR_EMAIL before running. Falls back to the existing pubmed_abstracts.csv if the API is unavailable.

In [ ]:
# ── 4a. Pick the top focus terms to query ───────────────────────────────────
# We use the 15 most frequent MedQuAD focus areas as our search terms.
# This keeps the PubMed corpus topically aligned with the QA data.
TOP_N_TERMS  = 15
ABSTRACTS_PER_TERM = 200      # 15 * 200 = 3 000 abstracts max
YOUR_EMAIL   = "your.email@university.edu"   # ← replace with your real email

top_terms = mq["focus"].value_counts().head(TOP_N_TERMS).index.tolist()
print(f"Top {TOP_N_TERMS} MedQuAD focus terms used as PubMed queries:")
for t in top_terms:
    print(f"  • {t}")


In [ ]:
# ── 4b. Fetch from NCBI (skipped if cache exists) ───────────────────────────
from Bio import Entrez

CACHE_PATH = PROC_DIR / "pubmed_fetched.csv"

def fetch_pubmed(query: str, max_results: int, email: str) -> list[dict]:
    """Fetch PubMed records for a query; return list of dicts."""
    Entrez.email = email
    # Step 1: search for PMIDs
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    search  = Entrez.read(handle); handle.close()
    pmids   = search["IdList"]
    if not pmids:
        return []
    # Step 2: fetch full records (XML) in one batch
    handle  = Entrez.efetch(db="pubmed", id=",".join(pmids),
                            rettype="xml", retmode="xml")
    records = Entrez.read(handle); handle.close()
    results = []
    for rec in records.get("PubmedArticle", []):
        try:
            art    = rec["MedlineCitation"]["Article"]
            pmid   = str(rec["MedlineCitation"]["PMID"])
            title  = str(art.get("ArticleTitle", ""))
            # Abstract may be split into labelled sections
            abs_obj = art.get("Abstract", {})
            abs_text_list = abs_obj.get("AbstractText", [])
            if isinstance(abs_text_list, list):
                abstract = " ".join(str(t) for t in abs_text_list)
            else:
                abstract = str(abs_text_list)
            # MeSH terms
            mesh_list = rec["MedlineCitation"].get("MeshHeadingList", [])
            mesh_terms = [str(m["DescriptorName"]) for m in mesh_list]
            results.append({"pmid": pmid, "title": title,
                            "abstract": abstract, "mesh_terms": mesh_terms,
                            "query_term": query})
        except Exception:
            continue
    return results

if CACHE_PATH.exists():
    pubmed_df = pd.read_csv(CACHE_PATH)
    pubmed_df["mesh_terms"] = pubmed_df["mesh_terms"].fillna("[]")
    print(f"📂 Loaded cached PubMed data: {pubmed_df.shape}")
else:
    print("🌐 Fetching from NCBI E-utilities …")
    all_records = []
    for term in top_terms:
        recs = fetch_pubmed(query=term, max_results=ABSTRACTS_PER_TERM, email=YOUR_EMAIL)
        all_records.extend(recs)
        print(f"  {term}: {len(recs)} records")
        time.sleep(0.35)   # stay within 3 req/sec rate limit
    pubmed_df = pd.DataFrame(all_records).drop_duplicates(subset=["pmid"])
    pubmed_df.to_csv(CACHE_PATH, index=False)
    print(f"\n✅ Saved {len(pubmed_df):,} records → {CACHE_PATH}")

print(f"\nPubMed DataFrame shape: {pubmed_df.shape}")
print(pubmed_df.head(3).to_string())


In [ ]:
# ── 4c. Fallback — use the pre-existing pubmed_abstracts.csv if API fetch fails
# The workspace already contains a 13 200-row PubMed CSV.  If Biopython is not
# installed or the API call fails, this cell blends the two sources so the
# rest of the notebook still runs.

import ast

def load_pubmed_csv_fallback(csv_path: Path) -> pd.DataFrame:
    """Parse the pre-existing pubmed_abstracts.csv into a tidy long-form df."""
    raw = pd.read_csv(csv_path)
    topic_cols = [c for c in raw.columns
                  if not c.endswith("_links") and c != "Unnamed: 0"]
    records = []
    for topic in topic_cols:
        for _, row in raw.iterrows():
            val = row[topic]
            if pd.isna(val):
                continue
            try:
                parsed = ast.literal_eval(str(val))
                if isinstance(parsed, tuple) and len(parsed) == 2:
                    abs_raw, title = parsed
                    abstract = " ".join(abs_raw) if isinstance(abs_raw, list) else str(abs_raw)
                else:
                    continue
            except Exception:
                continue
            records.append({"pmid": f"legacy_{topic}_{_}",
                            "title": title, "abstract": abstract,
                            "mesh_terms": "[]", "query_term": topic})
    return pd.DataFrame(records)

if pubmed_df.empty or len(pubmed_df) < 100:
    print("⚠️  API fetch empty or too small — falling back to pubmed_abstracts.csv")
    pubmed_df = load_pubmed_csv_fallback(PUBMED_CSV)
    print(f"Fallback loaded: {pubmed_df.shape}")
else:
    print(f"✅  Using API-fetched PubMed data ({len(pubmed_df):,} rows)")


---
## 5. PubMed — EDA

- Abstract length distribution
- MedQuAD focus term overlap check
- Top content word frequency (stopwords removed)

In [ ]:
# ── 5a. Abstract length distribution ────────────────────────────────────────
pubmed_df["ab_words"] = pubmed_df["abstract"].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(pubmed_df["ab_words"], bins=50, color="#9C27B0", edgecolor="white", alpha=0.85)
axes[0].axvline(pubmed_df["ab_words"].median(), color="red", linestyle="--",
                label=f"median = {pubmed_df['ab_words'].median():.0f}")
axes[0].set_title("PubMed Abstract Length (word count)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Words"); axes[0].set_ylabel("Frequency")
axes[0].legend()

# Box-plot by query term (top 10 terms for readability)
top10 = pubmed_df["query_term"].value_counts().head(10).index
subset = pubmed_df[pubmed_df["query_term"].isin(top10)]
sns.boxplot(data=subset, x="query_term", y="ab_words", palette="Set2", ax=axes[1])
axes[1].set_title("Abstract Length by Top-10 Query Terms", fontsize=12, fontweight="bold")
axes[1].set_xlabel(""); axes[1].tick_params(axis="x", rotation=40)

plt.suptitle("PubMed Abstract Length Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PROC_DIR / "pubmed_length_dist.png", dpi=150, bbox_inches="tight")
plt.show()

print("Abstract length stats (words):")
print(pubmed_df["ab_words"].describe().round(1))


In [ ]:
# ── 5b. Overlap check ────────────────────────────────────────────────────────
# For each unique MedQuAD focus term, check whether at least one PubMed abstract
# contains that term (case-insensitive substring match).
# High overlap = the two corpora are topically aligned (good for our pipeline).

unique_focus = mq["focus"].dropna().unique()
pubmed_text  = " ".join(pubmed_df["abstract"].fillna("").str.lower().tolist())

hit   = [t for t in unique_focus if t.lower() in pubmed_text]
miss  = [t for t in unique_focus if t.lower() not in pubmed_text]

print(f"MedQuAD unique focus areas : {len(unique_focus)}")
print(f"  ✅  With ≥1 PubMed match  : {len(hit)}  ({len(hit)/len(unique_focus)*100:.1f}%)")
print(f"  ❌  No PubMed match       : {len(miss)}  ({len(miss)/len(unique_focus)*100:.1f}%)")

print(f"\nSample un-matched terms (first 10):")
for t in miss[:10]:
    print(f"  • {t}")


In [ ]:
# ── 5c. Top terms word frequency (excluding stopwords) ──────────────────────
# This is a quick proxy for the vocabulary the NER model needs to handle.

all_tokens = []
for abstract in pubmed_df["abstract"].dropna():
    tokens = word_tokenize(abstract.lower())
    all_tokens.extend([t for t in tokens if t.isalpha() and t not in STOP_WORDS and len(t) > 2])

top_50 = Counter(all_tokens).most_common(50)
top_20_df = pd.DataFrame(top_50[:20], columns=["term", "count"])

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(data=top_20_df, x="term", y="count", palette="magma", ax=ax)
ax.set_title("Top-20 Content Words in PubMed Abstracts (stopwords removed)",
             fontsize=12, fontweight="bold")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig(PROC_DIR / "pubmed_top_terms.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop 20 terms:")
for term, cnt in top_50[:20]:
    print(f"  {term:<25} {cnt:,}")


---
## 6. Preprocessing

1. Text cleaning (HTML decode, tag strip, URL remove, lowercase) applied to both datasets
2. Drop malformed rows
3. 80/10/10 stratified split by question_type
4. Save CSV + Parquet

In [ ]:
# ── 6a. Text cleaning function ──────────────────────────────────────────────
import html

def clean_text(text: str) -> str:
    """
    Normalise a raw medical text string.

    Steps
    -----
    1. Decode HTML entities  (&amp; → & etc.)
    2. Strip HTML / XML tags
    3. Remove residual URLs
    4. Collapse whitespace / strip leading-trailing spaces
    5. Lowercase
    """
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))                          # &amp; → &
    text = re.sub(r"<[^>]+>", " ", text)                    # remove tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)    # remove URLs
    text = re.sub(r"\s+", " ", text).strip()                # normalise space
    text = text.lower()                                      # lowercase
    return text

# Smoke-test
sample_dirty = "<b>Glaucoma</b> is a &amp; group of diseases. See https://nih.gov"
print("Before:", sample_dirty)
print("After :", clean_text(sample_dirty))


In [ ]:
# ── 6b. Apply cleaning to both datasets ─────────────────────────────────────
mq["question_clean"] = mq["question"].apply(clean_text)
mq["answer_clean"]   = mq["answer"].apply(clean_text)

pubmed_df["abstract_clean"] = pubmed_df["abstract"].apply(clean_text)
pubmed_df["title_clean"]    = pubmed_df["title"].apply(clean_text)

print(f"MedQuAD  cleaned: {len(mq):,} rows")
print(f"PubMed   cleaned: {len(pubmed_df):,} rows")

# Quick sanity: before vs after length should be similar
mq["q_words_clean"] = mq["question_clean"].apply(lambda x: len(x.split()))
delta = (mq["q_words"] - mq["q_words_clean"]).describe()
print("\nWord-count delta Q (raw − clean) — should be near 0:")
print(delta.round(2))


In [ ]:
# ── 6c. Drop malformed rows ──────────────────────────────────────────────────
before = len(mq)
mq_clean = mq.dropna(subset=["answer"]).copy()
mq_clean = mq_clean[mq_clean["answer_clean"].str.strip() != ""].copy()
print(f"Dropped {before - len(mq_clean)} malformed rows  ({before} → {len(mq_clean)})")


In [ ]:
# ── 6d. Stratified 80 / 10 / 10 split ───────────────────────────────────────
# Stratify by question_type so every intent class is proportionally represented
# in all three splits — critical for unbiased NLU evaluation.

train_df, temp_df = train_test_split(
    mq_clean, test_size=0.20, stratify=mq_clean["question_type"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["question_type"], random_state=42
)

print("Split sizes:")
print(f"  Train : {len(train_df):,}  ({len(train_df)/len(mq_clean)*100:.1f}%)")
print(f"  Val   : {len(val_df):,}  ({len(val_df)/len(mq_clean)*100:.1f}%)")
print(f"  Test  : {len(test_df):,}  ({len(test_df)/len(mq_clean)*100:.1f}%)")

# Verify stratification holds
print("\nquestion_type proportions (train vs test — should be similar):")
prop_train = train_df["question_type"].value_counts(normalize=True).round(3)
prop_test  = test_df["question_type"].value_counts(normalize=True).round(3)
print(pd.DataFrame({"train": prop_train, "test": prop_test}))


In [ ]:
# ── 6e. Save to CSV & Parquet ────────────────────────────────────────────────
# CSV for readability, Parquet for fast loading in training scripts.

train_df.to_csv(PROC_DIR / "mq_train.csv", index=False)
val_df.to_csv(  PROC_DIR / "mq_val.csv",   index=False)
test_df.to_csv( PROC_DIR / "mq_test.csv",  index=False)

train_df.to_parquet(PROC_DIR / "mq_train.parquet", index=False)
val_df.to_parquet(  PROC_DIR / "mq_val.parquet",   index=False)
test_df.to_parquet( PROC_DIR / "mq_test.parquet",  index=False)

pubmed_df.to_csv(    PROC_DIR / "pubmed_clean.csv",     index=False)
pubmed_df.to_parquet(PROC_DIR / "pubmed_clean.parquet", index=False)

mq_clean.to_csv(    PROC_DIR / "mq_full_clean.csv",     index=False)
mq_clean.to_parquet(PROC_DIR / "mq_full_clean.parquet", index=False)

print("✅  Saved files:")
for f in sorted(PROC_DIR.glob("*.csv")) + sorted(PROC_DIR.glob("*.parquet")):
    print(f"  {f.name}  ({f.stat().st_size // 1024:,} KB)")


---
## 7. Baseline NER — scispaCy en_ner_bc5cdr_md

Runs BC5CDR (chemical + disease) over 30 sample MedQuAD questions. Compares extracted entities to ground-truth ocus. Regex fallback if scispaCy is not installed.

In [ ]:
# ── 7a. Load the model ───────────────────────────────────────────────────────
try:
    import spacy
    nlp_ner = spacy.load("en_ner_bc5cdr_md")
    NER_BACKEND = "scispacy_bc5cdr"
    print("✅  Loaded scispaCy en_ner_bc5cdr_md")
except (ImportError, OSError):
    print("⚠️  scispaCy not available — using regex fallback NER")
    nlp_ner = None
    NER_BACKEND = "regex_fallback"


In [ ]:
# ── 7b. Helper: extract entities from a question ────────────────────────────
def extract_entities(text: str) -> list[str]:
    """Return a list of entity strings found in text."""
    if nlp_ner is not None:
        doc = nlp_ner(str(text))
        return [ent.text.lower() for ent in doc.ents]
    else:
        # Regex fallback: capitalised multi-word noun phrases are likely entities
        return [m.group(0).lower()
                for m in re.finditer(r"[A-Z][a-z]+(?: [A-Z][a-z]+)*", str(text))]

def overlap_score(entities: list[str], ground_truth: str) -> float:
    """
    Rough recall-style overlap: fraction of GT tokens that appear in any
    extracted entity.  Returns 0.0 – 1.0.
    """
    if not entities or pd.isna(ground_truth):
        return 0.0
    gt_tokens = set(str(ground_truth).lower().split())
    ent_tokens = set(" ".join(entities).split())
    matches = gt_tokens & ent_tokens
    return len(matches) / len(gt_tokens)


In [ ]:
# ── 7c. Run NER on 30 sample questions ──────────────────────────────────────
# We sample 30 rows evenly across question types for a representative pass.
sample_30 = mq_clean.groupby("question_type", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 3), random_state=42)
).head(30).reset_index(drop=True)

results = []
for _, row in sample_30.iterrows():
    entities = extract_entities(row["question"])
    score    = overlap_score(entities, row["focus"])
    results.append({
        "question"    : row["question"],
        "focus_gt"    : row["focus"],
        "extracted"   : entities,
        "overlap"     : round(score, 2),
        "question_type": row["question_type"],
    })

ner_df = pd.DataFrame(results)

print(f"NER backend: {NER_BACKEND}")
print(f"Samples    : {len(ner_df)}")
print(f"Mean overlap score: {ner_df['overlap'].mean():.3f}\n")

# Print table
print(f"{'Question':<60} {'GT Focus':<25} {'Entities Found':<40} {'Overlap':>7}")
print("-" * 140)
for _, row in ner_df.iterrows():
    q      = str(row["question"])[:58]
    focus  = str(row["focus_gt"])[:23]
    ents   = str(row["extracted"])[:38]
    score  = row["overlap"]
    print(f"{q:<60} {focus:<25} {ents:<40} {score:>7.2f}")


In [ ]:
# ── 7d. Overlap score distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of overlap scores
axes[0].hist(ner_df["overlap"], bins=10, range=(0, 1),
             color="#FF5722", edgecolor="white", alpha=0.85)
axes[0].set_title(f"NER Overlap Score Distribution\n(backend: {NER_BACKEND})",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Overlap score (0 = no match, 1 = perfect)")
axes[0].set_ylabel("Count")
axes[0].axvline(ner_df["overlap"].mean(), color="blue", linestyle="--",
                label=f"mean = {ner_df['overlap'].mean():.2f}")
axes[0].legend()

# Per question-type breakdown
qt_overlap = ner_df.groupby("question_type")["overlap"].mean().sort_values()
sns.barplot(x=qt_overlap.values, y=qt_overlap.index, palette="rocket", ax=axes[1])
axes[1].set_title("Mean NER Overlap by Question Type", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Mean overlap score")

plt.tight_layout()
plt.savefig(PROC_DIR / "ner_overlap.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOverall mean overlap : {ner_df['overlap'].mean():.3f}")
print(f"Rows with score > 0  : {(ner_df['overlap'] > 0).sum()} / {len(ner_df)}")


---
## 8. Summary Report

In [ ]:
# ── 8. Print summary report ──────────────────────────────────────────────────
qt_counts = mq_clean["question_type"].value_counts()
sparse_classes = qt_counts[qt_counts < 50].to_dict()
dominant_class = qt_counts.idxmax()
imbalance_ratio = qt_counts.max() / max(qt_counts.min(), 1)

overlap_pct = len(hit) / max(len(unique_focus), 1) * 100

print("=" * 70)
print("  PROGRESS CHECKPOINT — SUMMARY REPORT")
print("=" * 70)

print(f"""
DATASET SIZES
  MedQuAD (after cleaning) : {len(mq_clean):,} QA pairs
    ├─ Train               : {len(train_df):,}
    ├─ Validation          : {len(val_df):,}
    └─ Test                : {len(test_df):,}
  PubMed subset            : {len(pubmed_df):,} abstracts
  Unique focus areas       : {mq_clean['focus'].nunique():,}
  Unique sources           : {mq_clean['source'].nunique()}  {mq_clean['source'].unique().tolist()}

QUESTION TYPE DISTRIBUTION
  Dominant class : {dominant_class}  ({qt_counts.max():,} examples)
  Imbalance ratio: {imbalance_ratio:.1f}:1  (largest vs smallest class)
  Sparse classes : {sparse_classes if sparse_classes else "None — all classes have ≥ 50 examples ✅"}

CLASS BALANCE IMPLICATIONS FOR NLU STAGE
  The "{dominant_class}" class holds {qt_counts.max()/len(mq_clean)*100:.1f}% of all examples.
  A vanilla classifier will be biased toward it.
  Recommended actions:
    1. Use class_weight="balanced" in sklearn / class_weights in PyTorch.
    2. Consider upsampling minority classes with paraphrase augmentation.
    3. Evaluate per-class F1, not macro accuracy.

TOPICAL ALIGNMENT (MedQuAD ↔ PubMed)
  {overlap_pct:.1f}% of MedQuAD focus terms appear in ≥1 PubMed abstract.
  {"✅  Good alignment — fine-tuning on PubMed should help NLU." if overlap_pct >= 70
   else "⚠️  Moderate alignment — consider broadening PubMed queries."}

BASELINE NER (scispaCy en_ner_bc5cdr_md | fallback={NER_BACKEND})
  Mean overlap score on 30 sample questions : {ner_df['overlap'].mean():.3f}
  {"✅  Decent recall — BC5CDR covers disease/chemical entities well."
   if ner_df['overlap'].mean() >= 0.4
   else "⚠️  Low recall — model misses many focus entities."}

NEXT-WEEK NLU / NER PRIORITIES
  1. Fine-tune a sequence classifier (BioBERT / PubMedBERT) on the 80/10/10
     split for question-type intent detection.
  2. Improve NER: add a custom entity ruler for MedQuAD focus terms not covered
     by BC5CDR, or fine-tune on a MedQuAD-derived NER corpus.
  3. Cross-validate using stratified k-fold to get stable class-level F1 scores
     before moving to the NLG stage.
""")
print("=" * 70)
